In [ ]:
##1. Imports

#import functions

import librosa
import numpy as np
import matplotlib.pyplot as plt
import scipy

In [ ]:
##2. Load Audio [INSERT FILE PATH]
y, sr = librosa.load('Audios/Jazz_Trio_Sample.wav', sr=None)  # Load audio file, sr=None to preserve original sampling rate

#Note: y --> audio array, sr --> sampling rate
#Note to self: May see Py SoundFile failed. Trying audioread instead --> warning message, not an error

In [ ]:
##3. Tempo Detection

#In theory, BPM could be determined by detecting peaks in the audio signal, which correspond to beats. 
#I will start with a simple approach using the librosa library.

#Beat Detection
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
print(f"Estimated Tempo: {tempo} BPM")

#Alternative calcuation
beat_times = librosa.frames_to_time(beat_frames, sr=sr)
beat_intervals = np.diff(beat_times)

bpm_from_intervals = 60 / np.mean(beat_intervals)
print(f"BPM from intervals: [{bpm_from_intervals}] BPM")

In [ ]:
##4 Tempo Curve [EXPERIMENTAL]
onset_env = librosa.onset.onset_strength(y=y, sr=sr)
dtempo = librosa.feature.tempo(onset_envelope=onset_env, sr=sr, aggregate=None)

plt.figure(figsize=(12, 4))
plt.plot(librosa.times_like(dtempo), dtempo)
plt.title("Tempo over Time")
plt.xlabel("Time (s)")
plt.ylabel("BPM")
plt.show()

from scipy.ndimage import uniform_filter1d

dtempo_smooth = uniform_filter1d(dtempo, size=2000) # Adjust size for more or less smoothing

plt.figure(figsize=(12, 4))
plt.plot(librosa.times_like(dtempo), dtempo_smooth)
plt.title("Tempo over Time (Smoothed)")
plt.xlabel("Time (s)")
plt.ylabel("BPM")
plt.show()

In [ ]:
##5. Chromagram 
chromagram = librosa.feature.chroma_cqt(y=y, sr=sr)

#Note: We use cqt and not stft as cqt is logarithmic, which corresponds better to human perception of pitch.

print(chromagram.shape)
#12 --> corresponds with the 12 notes
#The second number is the number of time frames: the number of samplings/snapshots

tuning = librosa.estimate_tuning(y=y, sr=sr)
print(f"Tuning offset: {tuning} cents")

In [ ]:
##5A: Chromogram Heatmap
#Great for visualizing how the intensity of each pitch changes over time.
fig = plt.figure()
librosa.display.specshow(chromagram, sr=sr, x_axis = "time", y_axis = "chroma")
plt.colorbar()
plt.title("Chromagram Heatmap")
plt.show()

In [ ]:
##5B: Chromagram Barchart
#Great for visualiziing which pitches are most prominent in the audio.
chroma_mean = np.mean(chromagram, axis=1)
#Note: np.mean with axis=1 calculates the mean across the time frames for each of the 12 bins
pitch_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
plt.bar(pitch_names, chroma_mean)
plt.xlabel("Note")
plt.ylabel("Mean Intensity")
plt.title("Chromagram Barchart")
plt.show()

In [ ]:
##6. Key Detection (music21) 
from music21 import stream, key, note

from music21.analysis.discrete import KrumhanslSchmuckler
analyzer = KrumhanslSchmuckler()

s = stream.Stream()

for pitch_name, value in zip(pitch_names, chroma_mean):
    i = int(value * 100)
    while i > 0:
        n = note.Note(pitch_name)
        s.append(n)
        i -= 1


result = analyzer.getSolution(s)
print(f"Estimated Key: {result}")


#Can use following for alternative estimates:
for alt in result.alternateInterpretations[:4]:
    print(f"Alternative: {alt}")



# Note: the estimation is not exactly accurate all the time, especially for minor keys 
# where KrumhanslSchmuckler doesn't account for harmonics

In [ ]:
##7. Key Detection (Custom)

#My OWN implementation of a simple key estimation based on the chroma mean and the Krumhansl-Schmuckler profiles.
#Note: This is a very basic implementation and may not be as accurate as the one provided
# by music21, but it serves as a good exercise in understanding the key estimation process.

#May implement Dorian, Phrygian, Lydian, Mixolydian, Aeolian, and Locrian modes in the future as well.

c_major = [6.0, 1.0, 3.0, 1.0, 6.0, 3.0, 1.0, 6.0, 1.0, 3.0, 1.0, 3.0]
#           C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_natural_minor = [6.0, 1.0, 3.0, 3.0, 1.0, 3.0, 1.0, 6.0, 1.0, 3.0, 3.0, 1.0]
#                   C    C#   D    D#   E    F    F#   G    G#   A    A#   B  


c_harmonic_minor = [6.0, 1.0, 3.0, 3.0, 1.0, 3.0, 1.0, 6.0, 3.0, 1.0, 1.0, 3.0]
#                   C    C#   D    D#   E    F    F#   G    G#   A    A#   B


harmonic_minor_profiles = []
for i in range(12):
    harmonic_minor_profiles.append(np.roll(c_harmonic_minor, i))

major_profiles = []
for i in range(12):
    major_profiles.append(np.roll(c_major, i))

minor_profiles = []
for i in range(12):
    minor_profiles.append(np.roll(c_natural_minor, i))


major_pitch_names = ['C Major', 'C# Major', 'D Major', 'D# Major', 'E Major', 'F Major', 'F# Major', 'G Major', 'G# Major', 'A Major', 'A# Major', 'B Major']
minor_pitch_names = ['C Minor', 'C# Minor', 'D Minor', 'D# Minor', 'E Minor', 'F Minor', 'F# Minor', 'G Minor', 'G# Minor', 'A Minor', 'A# Minor', 'B Minor']
harmonic_pitch_names = ['C Harmonic Minor', 'C# Harmonic Minor', 'D Harmonic Minor', 'D# Harmonic Minor', 'E Harmonic Minor', 'F Harmonic Minor', 'F# Harmonic Minor', 'G Harmonic Minor', 'G# Harmonic Minor', 'A Harmonic Minor', 'A# Harmonic Minor', 'B Harmonic Minor']

All_names = major_pitch_names + minor_pitch_names + harmonic_pitch_names

scores = []
for profile in major_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])
for profile in minor_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])
for profile in harmonic_minor_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])

top3_indices = np.argsort(scores)[-3:][::-1]
for i in top3_indices:
    print(f"{All_names[i]}: {scores[i]:.4f}")

In [ ]:
##8. Summary Cell
print("=" * 40)
print("AUDIO ANALYSIS SUMMARY")
print("=" * 40)
print(f"Estimated Tempo: {tempo} BPM")
print(f"Tempo from intervals: {bpm_from_intervals:.2f} BPM")
print()
print(f"Krumhansl-Schmuckler Key: {result}")
for alt in result.alternateInterpretations[:4]:
    print(f"  Alternative: {alt}")
print()
print("Custom Key Estimation:")
for i in top3_indices:
    print(f"  {All_names[i]}: {scores[i]:.4f}")
print("=" * 40)

In [ ]:
##Extra: Time Signature [Experimental]

# pulse = librosa.beat.plp(y=y, sr=sr)
# pulse_times = librosa.frames_to_time(np.arange(len(pulse)), sr=sr)
# plt.figure(figsize=(12, 4))
# plt.plot(pulse_times, pulse)
# plt.xlim(0, 5)
# plt.title("Pulse Strength - First 5 seconds")
# plt.xlabel("Time (s)")
# plt.ylabel("Pulse Strength")
# plt.show()

#Note: This one is very difficult, may not use or implement for now.


In [ ]:
# #EXTRA: Detecting Harmonic Minors [Origional Code]

# c_harmonic_minor = [6.0, 1.0, 3.0, 3.0, 1.0, 3.0, 1.0, 6.0, 3.0, 1.0, 1.0, 3.0]
# #                   C    C#   D    D#   E    F    F#   G    G#   A    A#   B
# #Mirrors the weights for the minor scale, but with the raised 7th (harmonic minor) instead of the natural minor.

# harmonic_minor_profiles = []
# for i in range(12):
#     harmonic_minor_profiles.append(np.roll(c_harmonic_minor, i))

# scores = []
# for profile in harmonic_minor_profiles:
#     score = np.corrcoef(chroma_mean, profile)
#     scores.append(score[0, 1])


# harmonic_pitch_names = ['C Harmonic Minor', 'C# Harmonic Minor', 'D Harmonic Minor', 'D# Harmonic Minor', 'E Harmonic Minor', 'F Harmonic Minor', 'F# Harmonic Minor', 'G Harmonic Minor', 'G# Harmonic Minor', 'A Harmonic Minor', 'A# Harmonic Minor', 'B Harmonic Minor']

# highest_score_index = np.argmax(scores)
# estimated_key_harmonic_minor = harmonic_pitch_names[highest_score_index]
# print(f"Estimated Key (Harmonic Minor): {estimated_key_harmonic_minor}")